# Week 2 &mdash; Learning on Graphs

## Theory: Graphs, Adjacency Matrices, and Spectral Theory

**Geometric Deep Learning &mdash; From Foundations to Applications**

---

### Learning Objectives

1. Define graphs formally and represent them with **adjacency**, **degree**, and **Laplacian** matrices.
2. Understand the **graph Laplacian** as a discrete analogue of the continuous Laplace operator.
3. Derive the **graph Fourier transform** from the eigendecomposition of the Laplacian.
4. Define **spectral graph convolution** and explain why it generalises classical convolution.
5. Appreciate the computational obstacles that motivate the *spatial* message-passing view of Week 3.

---


## 1. Graphs and Their Matrix Representations

A **graph** is a pair $G = (V, E)$ with a finite vertex set $V = \{1, \dots, n\}$ and an edge set $E \subseteq V \times V$. We allow weighted edges via a weight function $w : E \to \mathbb{R}_{>0}$.

**Adjacency matrix.** $A \in \mathbb{R}^{n \times n}$ with
$$
A_{ij} = \begin{cases} w(i,j) & (i,j) \in E,\\ 0 & \text{otherwise.}\end{cases}
$$
For an undirected graph, $A = A^\top$.

**Degree matrix.** $D = \mathrm{diag}(d_1, \dots, d_n)$ where $d_i = \sum_j A_{ij}$ is the (weighted) degree of vertex $i$.

A **graph signal** is a vector $x \in \mathbb{R}^n$ assigning a scalar (or, more generally, a feature vector $X \in \mathbb{R}^{n \times d}$) to each vertex. The key symmetry of a graph is the relabelling of its nodes: the **permutation group** $S_n$. Any meaningful graph operator must be permutation-equivariant, i.e. commute with $X \mapsto PX$, $A \mapsto P A P^\top$ for permutation matrices $P$.


## 2. The Graph Laplacian

The central operator of spectral graph theory is the **combinatorial (unnormalised) Laplacian**
$$
L = D - A.
$$
It is the discrete counterpart of the continuous Laplacian $-\Delta = -\nabla^2$. Its action on a signal is
$$
(Lx)_i = \sum_{j} A_{ij}\,(x_i - x_j),
$$
a weighted sum of *differences* to neighbours &mdash; precisely a discrete second derivative.

**Key properties.**

- $L$ is **symmetric positive semi-definite**: for any $x$,
$$
x^\top L x = \tfrac{1}{2}\sum_{i,j} A_{ij}\,(x_i - x_j)^2 \;\geq\; 0.
$$
This quadratic form is the **Dirichlet energy**, measuring how non-smooth $x$ is across edges.
- The constant vector $\mathbf{1}$ lies in the kernel: $L\mathbf{1} = 0$, so $0$ is always an eigenvalue.
- The **multiplicity of the eigenvalue $0$** equals the number of connected components of $G$.

Two normalised variants are common:
$$
L_{\mathrm{sym}} = D^{-1/2} L D^{-1/2} = I - D^{-1/2} A D^{-1/2}, \qquad
L_{\mathrm{rw}} = D^{-1} L = I - D^{-1} A.
$$
The symmetric form $L_{\mathrm{sym}}$ has eigenvalues in $[0, 2]$ and is the version underlying the Graph Convolutional Network of Week 3.


## 3. The Spectral Theorem and the Graph Fourier Transform

Because $L$ is real-symmetric and PSD, the **spectral theorem** gives an orthonormal eigenbasis:
$$
L = U \Lambda U^\top, \qquad U = [u_1, \dots, u_n], \qquad \Lambda = \mathrm{diag}(\lambda_1, \dots, \lambda_n),
$$
with $0 = \lambda_1 \leq \lambda_2 \leq \cdots \leq \lambda_n$ and $U^\top U = I$.

By analogy with the classical setting &mdash; where the complex exponentials are eigenfunctions of the continuous Laplacian &mdash; the eigenvectors $u_k$ are the **graph Fourier modes**, and the eigenvalues $\lambda_k$ play the role of (squared) **frequencies**. Low $\lambda_k$ correspond to *smooth* modes; high $\lambda_k$ to *oscillatory* ones.

**Graph Fourier transform (GFT).**
$$
\hat{x} = U^\top x \quad (\text{analysis}), \qquad x = U \hat{x} \quad (\text{synthesis}).
$$
The coefficient $\hat{x}_k = \langle u_k, x\rangle$ measures the energy of $x$ at frequency $\lambda_k$.


## 4. Spectral Graph Convolution

In Euclidean signal processing, convolution becomes pointwise multiplication in the Fourier domain (the convolution theorem). We *define* graph convolution by analogy: a filter is a function $g_\theta(\Lambda)$ of the eigenvalues, and
$$
g_\theta \star x \;=\; U\, g_\theta(\Lambda)\, U^\top x \;=\; U\,\mathrm{diag}\!\big(g_\theta(\lambda_1), \dots, g_\theta(\lambda_n)\big)\,U^\top x.
$$

This is the foundational definition of **spectral GNNs** (Bruna et al., 2014). It is permutation-equivariant and, for any choice of $g_\theta$, a well-defined linear operator on graph signals. However, it has three serious drawbacks:

1. **Cost.** Forming $U$ requires a full eigendecomposition, $\mathcal{O}(n^3)$ &mdash; prohibitive for large graphs.
2. **Non-locality.** A general $g_\theta(\Lambda)$ produces filters with global support; a single layer mixes information across the entire graph.
3. **Basis dependence.** Filters learned on one graph's eigenbasis do not transfer to another graph.


## 5. Polynomial Filters: From Spectral to Spatial

The remedy is to restrict $g_\theta$ to be a **polynomial** of the Laplacian (Defferrard et al., 2016):
$$
g_\theta(L) = \sum_{k=0}^{K} \theta_k\, L^k.
$$
Two facts make this transformative:

- **Locality.** $(L^k)_{ij} = 0$ whenever the shortest-path distance between $i$ and $j$ exceeds $k$. Hence a degree-$K$ polynomial filter is *exactly $K$-localised*: it only mixes information within a $K$-hop neighbourhood.
- **No eigendecomposition.** $g_\theta(L)x = \sum_k \theta_k L^k x$ can be computed by repeated sparse matrix&ndash;vector products, at cost $\mathcal{O}(K|E|)$.

Using the **Chebyshev polynomials** $T_k$ for numerical stability gives the *ChebNet* filter
$$
g_\theta(L) = \sum_{k=0}^{K} \theta_k\, T_k(\tilde{L}), \qquad \tilde{L} = \tfrac{2}{\lambda_{\max}} L - I.
$$
Truncating to $K = 1$ and applying a renormalisation yields the celebrated **Graph Convolutional Network** propagation rule
$$
H^{(l+1)} = \sigma\!\big(\tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}\, H^{(l)} W^{(l)}\big), \qquad \tilde{A} = A + I,
$$
which we study in detail in Week 3. The polynomial perspective is the bridge from the *spectral* theory developed here to the *spatial* message-passing framework that follows.


## 6. Summary

- A graph is encoded by its **adjacency** $A$, **degree** $D$, and **Laplacian** $L = D - A$.
- The Laplacian is a PSD operator whose quadratic form is the **Dirichlet energy**; its kernel encodes connectivity.
- Its eigenbasis defines the **graph Fourier transform** and a notion of graph **frequency**.
- **Spectral convolution** $U g_\theta(\Lambda) U^\top$ generalises classical convolution but is costly, non-local, and basis-dependent.
- **Polynomial (Chebyshev) filters** restore locality and efficiency, and degenerate into the **GCN** rule &mdash; the link to spatial message passing.

### References

- Bruna, J., Zaremba, W., Szlam, A., &amp; LeCun, Y. (2014). *Spectral Networks and Locally Connected Networks on Graphs.* ICLR.
- Defferrard, M., Bresson, X., &amp; Vandergheynst, P. (2016). *Convolutional Neural Networks on Graphs with Fast Localized Spectral Filtering.* NeurIPS.
- Hamilton, W. L. (2020). *Graph Representation Learning.* Morgan &amp; Claypool. Chapters 1&ndash;3.

### Exercises

1. Prove $x^\top L x = \tfrac{1}{2}\sum_{ij} A_{ij}(x_i - x_j)^2$.
2. Show that the algebraic multiplicity of eigenvalue $0$ of $L$ equals the number of connected components.
3. Demonstrate that $(L^k)_{ij} = 0$ if the graph distance $d(i,j) > k$.
